# Step 3: Build SPC Features (with Horizon Mapping)
## Modified to add SPC features to column_step_map.json

This notebook builds the SPC metrology feature matrix and adds SPC feature mappings to column_step_map.json.

In [1]:
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

start_time = time.time()
print("=" * 60)
print("Step 3: Building SPC Features (with Horizon Mapping)")
print("=" * 60)

Step 3: Building SPC Features (with Horizon Mapping)


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


In [3]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
features_dir.mkdir(parents=True, exist_ok=True)

sentinel = features_dir / "03_spc.done"
force = False  # Set to True to force rebuild

if sentinel.exists() and not force:
    print(f"[OK] SPC features already built (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print("Building SPC features...")

[OK] SPC features already built (found outputs\features\03_spc.done)
  Set force=True to rebuild


In [4]:
# Check input file
spc_file = data_dir / "spc.csv"
if not spc_file.exists():
    raise FileNotFoundError(
        f"Could not find {spc_file}\n"
        f"Please ensure spc.csv is in the data/ directory."
    )

print(f"Loading SPC data from {spc_file}...")
spc = pd.read_csv(spc_file)
print(f"  Loaded {len(spc):,} rows")

Loading SPC data from data\spc.csv...


C:\Users\nickp\AppData\Local\Temp\ipykernel_2360\2076613821.py:10: DtypeWarning: Columns (0: PROCESS_POSITION) have mixed types. Specify dtype option on import or set low_memory=False.
  spc = pd.read_csv(spc_file)


  Loaded 1,620,882 rows


In [5]:
# Build mean features
print("Building SPC mean features...")
spc['mean_feature'] = (
    spc['PARAMETER'] + "__" +
    spc['PROCESS_STEP'] + "__" +
    spc['METRO_STEP'] + "__SPC_MEAN"
)
mean_pivot = spc.pivot_table(
    index='WAFER_SCRIBE',
    columns='mean_feature',
    values='MeanValue',
    aggfunc='mean'
)
print(f"  Mean features shape: {mean_pivot.shape}")

Building SPC mean features...
  Mean features shape: (18669, 1736)


In [6]:
# Build stdev features
print("Building SPC stdev features...")
spc['std_feature'] = (
    spc['PARAMETER'] + "__" +
    spc['PROCESS_STEP'] + "__" +
    spc['METRO_STEP'] + "__SPC_STD"
)
std_pivot = spc.pivot_table(
    index='WAFER_SCRIBE',
    columns='std_feature',
    values='StdDev',
    aggfunc='mean'
)
print(f"  Stdev features shape: {std_pivot.shape}")

Building SPC stdev features...
  Stdev features shape: (18669, 1736)


In [7]:
# Build position features
print("Building process position features...")
spc['pos_feature'] = spc['PROCESS_STEP'] + "__POSITION"
pos_pivot = spc.pivot_table(
    index='WAFER_SCRIBE',
    columns='pos_feature',
    values='PROCESS_POSITION',
    aggfunc='first'
)
print(f"  Position features shape: {pos_pivot.shape}")

Building process position features...
  Position features shape: (18669, 39)


In [8]:
# Combine all SPC features
print("Combining SPC features...")
spc_features = pd.concat([mean_pivot, std_pivot, pos_pivot], axis=1)
print(f"  Combined shape: {spc_features.shape}")

Combining SPC features...
  Combined shape: (18669, 3511)


In [9]:
# Add missingness indicators
print("Adding missingness indicators...")
missing_cols = {}
for col in tqdm(spc_features.columns, desc="Missingness"):
    if spc_features[col].isna().any():
        missing_cols[col + "__MISSING"] = spc_features[col].isna().astype(int)

if missing_cols:
    missing_df = pd.DataFrame(missing_cols, index=spc_features.index)
    spc_features = pd.concat([spc_features, missing_df], axis=1)
    print(f"  Added {len(missing_cols)} missingness indicators")

Adding missingness indicators...


Missingness: 100%|██████████| 3511/3511 [00:01<00:00, 2067.35it/s]


  Added 3507 missingness indicators


In [10]:
# Impute with per-column median
print("Imputing missing values with per-column median...")
for col in tqdm(spc_features.columns, desc="Imputing"):
    if spc_features[col].isna().any() and not col.endswith("__MISSING"):
        if spc_features[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            spc_features[col] = spc_features[col].fillna(spc_features[col].median())
        elif spc_features[col].dtype == 'object' or spc_features[col].dtype.name == 'string':
            mode_val = spc_features[col].mode()[0] if len(
                spc_features[col].mode()) > 0 else 'UNKNOWN'
            spc_features[col] = spc_features[col].fillna(mode_val)

Imputing missing values with per-column median...


Imputing: 100%|██████████| 7018/7018 [00:03<00:00, 1875.93it/s]


In [11]:
# Drop near-zero variance columns
print("Dropping near-zero variance columns (>95% identical)...")
initial_cols = len(spc_features.columns)
cols_to_drop = []
for col in tqdm(spc_features.columns, desc="Variance check"):
    if spc_features[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
        value_counts = spc_features[col].value_counts()
        if len(value_counts) > 0 and value_counts.iloc[0] / len(spc_features) > 0.95:
            cols_to_drop.append(col)

spc_features.drop(columns=cols_to_drop, inplace=True)
print(f"  Dropped {len(cols_to_drop)} columns ({initial_cols} -> {len(spc_features.columns)})")

Dropping near-zero variance columns (>95% identical)...


Variance check: 100%|██████████| 7018/7018 [00:01<00:00, 4382.26it/s]


  Dropped 6611 columns (7018 -> 407)


## NEW: Add SPC Features to Column-Step Mapping

SPC metrology is taken after a PROCESS_STEP. We map each SPC feature to the SeqNo of its PROCESS_STEP and add to column_step_map.json.

In [12]:
# Load step_seq.csv to map PROCESS_STEP to SeqNo
print("\n" + "=" * 60)
print("Adding SPC features to column-step mapping...")
print("=" * 60)

step_seq_file = data_dir / "step_seq.csv"
if not step_seq_file.exists():
    print(f"\nWARNING: {step_seq_file} not found.")
    print("Cannot create SPC feature mappings without step sequence data.")
    step_to_seqno = {}
else:
    step_seq_df = pd.read_csv(step_seq_file)
    step_to_seqno = dict(zip(step_seq_df['STEP'], step_seq_df['SeqNo']))
    print(f"  Loaded step_seq.csv with {len(step_to_seqno)} steps")


Adding SPC features to column-step mapping...
  Loaded step_seq.csv with 240 steps


In [13]:
# Load existing column_step_map from sensor features (if it exists)
column_map_file = features_dir / "column_step_map.json"

if column_map_file.exists():
    print(f"\nLoading existing column_step_map.json...")
    with open(column_map_file, 'r') as f:
        column_step_map = json.load(f)
    print(f"  Loaded {len(column_step_map)} existing sensor feature mappings")
else:
    print(f"\nWARNING: column_step_map.json not found. Creating new one.")
    column_step_map = {}


Loading existing column_step_map.json...
  Loaded 9888 existing sensor feature mappings


In [14]:
# Add SPC feature mappings
spc_mappings_added = 0

for col in spc_features.columns:
    # Extract PROCESS_STEP from column name
    # Format: PARAMETER__PROCESS_STEP__METRO_STEP__SPC_MEAN or similar
    parts = col.split('__')
    
    if len(parts) >= 2:
        process_step = parts[1]  # PROCESS_STEP is second component
        
        # Look up SeqNo for this PROCESS_STEP
        if process_step in step_to_seqno:
            column_step_map[col] = int(step_to_seqno[process_step])
            spc_mappings_added += 1
        else:
            # If step not found, assign 0 (always included)
            column_step_map[col] = 0
            if step_to_seqno:  # Only warn if we have step_seq data
                print(f"  WARNING: Process step '{process_step}' not found in step_seq.csv")
    else:
        # If column doesn't follow expected format, assign 0
        column_step_map[col] = 0

print(f"\nAdded {spc_mappings_added} SPC feature mappings to column_step_map")
print(f"  Total mappings: {len(column_step_map)}")


Added 361 SPC feature mappings to column_step_map
  Total mappings: 10295


In [15]:
# Save updated column_step_map.json
with open(column_map_file, 'w') as f:
    json.dump(column_step_map, f, indent=2)

print(f"\n[OK] Updated column_step_map.json with {len(column_step_map)} total entries")

# Show sample SPC mappings
print("\nSample SPC column-to-step mappings:")
spc_cols = [col for col in column_step_map.keys() if '__SPC' in col or '__POSITION' in col]
for i, col in enumerate(spc_cols[:5]):
    print(f"  {col}: SeqNo={column_step_map[col]}")


[OK] Updated column_step_map.json with 10295 total entries

Sample SPC column-to-step mappings:
  CA1_IN_S01_AVG__PV0003__PV0003__SPC_MEAN: SeqNo=135
  CA1_IN_S02_AVG__PV0003__PV0003__SPC_MEAN: SeqNo=135
  CA2_OUT_S01_AVG__PV0003__PV0003__SPC_MEAN: SeqNo=135
  CA2_OUT_S02_AVG__PV0003__PV0003__SPC_MEAN: SeqNo=135
  CH_PRESS_S06_AVG__PV0003__PV0003__SPC_MEAN: SeqNo=135


In [16]:
# Calculate sparsity
total_cells = spc_features.shape[0] * spc_features.shape[1]
missing_cells = spc_features.isna().sum().sum()
sparsity = 100 * missing_cells / total_cells if total_cells > 0 else 0

# Save SPC features
output_file = features_dir / "spc_features.parquet"
print(f"\nSaving SPC features to {output_file}...")
spc_features.to_parquet(output_file)

# Print statistics
print("\n" + "=" * 60)
print("SPC Feature Statistics:")
print("=" * 60)
print(f"Total wafers: {len(spc_features)}")
print(f"Total features: {len(spc_features.columns)}")
print(f"Sparsity: {sparsity:.2f}%")
print(f"Memory usage: {spc_features.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] SPC features complete in {elapsed:.2f}s")
print("=" * 60)


Saving SPC features to outputs\features\spc_features.parquet...

SPC Feature Statistics:
Total wafers: 18669
Total features: 407
Sparsity: 8.82%
Memory usage: 62.36 MB

[OK] SPC features complete in 42.26s
